# Day07：Project Understanding

## Goal

扫描真实 Unity 工程，建立模块、脚本、Scene、Prefab 和轻量依赖线索，并把结果保存为可复用的 `project_context.json`。Day07 继续使用 `day06` 作为活跃源码目录。

## Setup

默认测试工程为 `D:\Unity\Unity_Project\CodingAgentTest`。扫描过程只读取 Unity 工程，因此 Unity 编辑器可以保持打开。也可通过环境变量 `UNITY_TEST_PROJECT_PATH` 切换工程。

In [1]:
import json
import os
import sys
from pathlib import Path

workspace = Path.cwd().resolve()
if workspace.name == 'day07':
    workspace = workspace.parent
day06_path = workspace / 'day06'
if not day06_path.is_dir():
    raise RuntimeError(f'未找到 day06 源码目录: {day06_path}')
if str(day06_path) not in sys.path:
    sys.path.insert(0, str(day06_path))

unity_project_path = Path(os.getenv(
    'UNITY_TEST_PROJECT_PATH',
    r'D:\Unity\Unity_Project\CodingAgentTest'
)).resolve()
context_path = day06_path / 'memory' / 'project_context.json'
print('Unity 工程:', unity_project_path)
print('上下文文件:', context_path)

Unity 工程: D:\Unity\Unity_Project\CodingAgentTest
上下文文件: D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\memory\project_context.json


## Steps

### 1. 扫描并持久化工程上下文

In [2]:
from memory.project_context import ProjectContextStore
from tools.project_scanner import UnityProjectScanner
from workflow.project_understanding import ProjectUnderstandingNode

store = ProjectContextStore(context_path)
node = ProjectUnderstandingNode(
    UnityProjectScanner(unity_project_path),
    store,
)
result = node.run({'agent_history': []})
if result['project_context_status'] != 'success':
    raise RuntimeError(result['project_context_error'])
project_context = result['project_context']
print(json.dumps(project_context['summary'], ensure_ascii=False, indent=2))

[Project Understanding]扫描完成:D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\memory\project_context.json
{
  "assets": 7,
  "scripts": 5,
  "scenes": 1,
  "prefabs": 0,
  "modules": 3,
  "declarations": 20,
  "scan_errors": 0
}


### 2. 查看有界的工程地图

只展示规划最有用的结构信息，避免输出完整资源内容。Prefab 数量为 0 也是合法结果。

In [3]:
project_map = {
    'modules': project_context['modules'],
    'scripts': [
        {
            'path': script['path'],
            'namespace': script['namespace'],
            'declarations': script['declarations'],
            'dependency_hints': script['dependency_hints'],
        }
        for script in project_context['scripts']
    ],
    'scenes': project_context['scenes'],
    'prefabs': project_context['prefabs'],
}
print(json.dumps(project_map, ensure_ascii=False, indent=2))

{
  "modules": [
    {
      "name": "Generated",
      "path": "Assets/Generated",
      "assets": 5,
      "scripts": 5,
      "scenes": 0,
      "prefabs": 0
    },
    {
      "name": "Resources",
      "path": "Assets/Resources",
      "assets": 1,
      "scripts": 0,
      "scenes": 0,
      "prefabs": 0
    },
    {
      "name": "Scenes",
      "path": "Assets/Scenes",
      "assets": 1,
      "scripts": 0,
      "scenes": 1,
      "prefabs": 0
    }
  ],
  "scripts": [
    {
      "path": "Assets/Generated/InventoryController.cs",
      "namespace": "",
      "declarations": [
        {
          "kind": "class",
          "name": "InventoryController",
          "full_name": "InventoryController",
          "base_types": [
            "MonoBehaviour"
          ]
        }
      ],
      "dependency_hints": {
        "using_namespaces": [
          "InventorySystem",
          "System",
          "TMPro",
          "UnityEngine",
          "UnityEngine.EventSystems",
         

## Checks

验证真实工程已被识别、上下文可重新读取，并且扫描期间没有错误。

In [4]:
loaded_context = store.load()
summary = loaded_context['summary']
assert loaded_context['schema_version'] == 1
assert loaded_context['project']['name'] == 'CodingAgentTest'
assert summary['assets'] >= 1
assert summary['scripts'] >= 1
assert summary['scenes'] >= 1
assert summary['scan_errors'] == 0
assert context_path.is_file()
assert result['current_agent'] == 'project_understanding'
print('PASS: Day07 真实 Unity Project Understanding 集成验收通过')

PASS: Day07 真实 Unity Project Understanding 集成验收通过


## Next Steps

- Architecture 与 File Planner 会读取这份工程上下文，优先复用已有类和命名空间。
- Day08 再基于这些确定性扫描结果建立更完整的代码依赖图，不在 Day07 提前扩大范围。